<a href="https://colab.research.google.com/github/RafihaikalP/BigData26_A_2411532002_Rafi-Haikal-Pratama/blob/main/Praktikum2/BD_A_P02_2411532002_Rafi_Haikal_Pratama.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install faker -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 25.3 MB/s eta 0:00:00


Perintah ini dijalankan di awal untuk menginstal library
 Faker yang tidak tersedia secara default di Google Colab. Tanda ! di depan artinya perintah ini dijalankan langsung ke terminal sistem, bukan ke Python. Flag -q berarti "quiet" — proses instalasi berjalan tanpa menampilkan terlalu banyak teks di layar.

# **K-1 — Import Library dan Inisialisasi**

In [2]:
import numpy as np
import pandas as pd
from faker import Faker
import random

Baris-baris ini mengimpor semua library yang akan dipakai sepanjang praktikum. NumPy diimpor dengan alias np — digunakan untuk operasi angka acak dan nilai np.nan (representasi nilai kosong). Pandas diimpor dengan alias pd — ini adalah library utama untuk membaca, mengolah, dan menyimpan data dalam bentuk tabel (DataFrame). Faker diimpor untuk membuat data palsu yang realistis seperti nama orang, nama kota, dan tanggal. Random adalah library bawaan Python untuk memilih nilai secara acak dari sebuah daftar

# K-2 — Membuat Dataset **Sintetis**

In [3]:
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
fake = Faker("id_ID")
Faker.seed(SEED)

N = 500
kategori_produk = ["Elektronik", "Fashion", "Kesehatan", "Rumah Tangga", "Olahraga", "Buku"]
metode_bayar = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]

rows = []
for i in range(1, N + 1):
    trx_id = f"TRX{i:05d}"
    nama_pelanggan = fake.name()
    produk = fake.word().capitalize() + " " + random.choice(["Pro", "Lite", "Max", "Basic", ""])
    kategori = random.choice(kategori_produk)
    harga_dasar = random.choice([15000, 25000, 50000, 75000, 120000, 250000, 500000, 1200000])
    qty = random.randint(1, 5)

    # Variasi format harga
    harga_variants = [
        str(harga_dasar),
        f"Rp{harga_dasar:,}".replace(",", "."),
        f"{harga_dasar}.0",
        f" {harga_dasar} ",
    ]
    harga = random.choice(harga_variants)

    # Variasi format tanggal
    tgl = fake.date_between(start_date="-90d", end_date="today")
    tgl_variants = [
        tgl.strftime("%Y-%m-%d"),
        tgl.strftime("%d/%m/%Y"),
        tgl.strftime("%d-%m-%Y")
    ]
    tanggal = random.choice(tgl_variants)

    metode = random.choice(metode_bayar)
    if random.random() < 0.3:
        metode = metode.lower()
    if random.random() < 0.2:
        kategori = kategori.upper() + " "

    kota = fake.city()
    rating = random.choice([1, 2, 3, 4, 5, None, None])

    rows.append({
        "transaction_id": trx_id,
        "customer_name": nama_pelanggan,
        "product_name": produk.strip(),
        "category": kategori,
        "price": harga,
        "quantity": qty,
        "payment_method": metode,
        "transaction_date": tanggal,
        "shipping_city": kota,
        "rating": rating,
    })

df = pd.DataFrame(rows)

# Sisipkan missing value
for col, frac in [("customer_name", 0.02), ("shipping_city", 0.03), ("payment_method", 0.015)]:
    idx = df.sample(frac=frac, random_state=SEED).index
    df.loc[idx, col] = np.nan

# Duplikasi 15 baris
dup_rows = df.sample(n=15, random_state=SEED)
df = pd.concat([df, dup_rows], ignore_index=True)
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

df.to_csv("transaksi_mentah.csv", index=False)
print("Jumlah baris:", len(df))

Jumlah baris: 515


Keseluruhan cell K-2 ini bertujuan mensimulasikan proses data acquisition — yaitu bagaimana data mentah datang dari sumber transaksional di dunia nyata. Proses dimulai dengan menetapkan SEED = 42 dan memanggil np.random.seed(), random.seed(), serta Faker.seed() secara bersamaan. Ketiga pemanggilan ini wajib dilakukan sekaligus karena NumPy, modul random bawaan Python, dan Faker adalah tiga generator acak yang benar-benar terpisah — jika salah satu tidak diatur, hasilnya tidak akan identik saat dijalankan ulang. Dengan seed yang sama, setiap mahasiswa yang menjalankan kode ini akan mendapatkan dataset yang persis sama.

Setelah inisialisasi, loop for i in range(1, N+1) berjalan sebanyak 500 kali untuk membuat 500 baris transaksi. Di dalam setiap putaran loop, seluruh kolom untuk satu transaksi dibuat sekaligus. ID transaksi dibuat dengan format TRX00001 hingga TRX00500 menggunakan format string {i:05d}. Nama pelanggan, nama produk, dan nama kota dihasilkan oleh Faker secara acak agar terdengar natural.

Yang paling penting di bagian ini adalah penyisipan masalah data mentah secara sengaja. Untuk kolom harga, satu nilai harga yang sama sengaja disimpan dalam empat format berbeda — angka biasa (50000), format rupiah dengan titik ribuan (Rp50.000), format desimal (50000.0), dan angka dengan spasi di sekelilingnya (50000) — lalu salah satu dipilih secara acak. Hal serupa dilakukan untuk tanggal yang disimpan dalam tiga format berbeda: ISO (2026-07-11), slash (11/07/2026), dan dash (11-07-2026). Untuk metode pembayaran dan kategori, inkonsistensi kapitalisasi disisipkan secara probabilistik — ada 30% kemungkinan metode pembayaran ditulis huruf kecil semua, dan 20% kemungkinan kategori ditulis huruf besar semua dengan spasi di belakangnya. Untuk rating, nilai None disertakan dua kali dalam daftar pilihan sehingga peluang rating kosong lebih besar, mensimulasikan kondisi nyata di mana banyak pembeli tidak mengisi rating. Semua variasi ini meniru permasalahan yang biasa muncul ketika data dikumpulkan dari banyak sistem atau banyak pengguna yang berbeda.

Setelah loop selesai, pd.DataFrame(rows) mengubah seluruh list transaksi menjadi satu tabel DataFrame. Kemudian missing value disisipkan secara terkendali menggunakan df.sample() — sekitar 2% baris kehilangan customer_name, 3% kehilangan shipping_city, dan 1.5% kehilangan payment_method. Terakhir, 15 baris dipilih secara acak lalu digandakan menggunakan pd.concat(), kemudian seluruh baris diacak urutannya dengan df.sample(frac=1) agar duplikat tidak terlihat berurutan. Hasilnya adalah dataset 515 baris yang penuh dengan masalah kualitas data — persis seperti data mentah yang akan ditemui dalam praktik nyata — dan disimpan sebagai transaksi_mentah.csv.

# K-3 — Deteksi dan Penanganan Missing **Value**

In [4]:
print(df.isnull().sum())

transaction_id        0
customer_name        20
product_name          0
category              0
price                 0
quantity              0
payment_method       16
transaction_date      0
shipping_city        30
rating              166
dtype: int64


df.isnull() menghasilkan DataFrame baru berisi nilai True atau False untuk setiap sel — True jika nilainya kosong, False jika tidak. .sum() menjumlahkan nilai True per kolom (karena True dihitung sebagai 1). Hasilnya adalah ringkasan berapa missing value yang ada di setiap kolom.

In [5]:
df = df.dropna(subset=["customer_name", "payment_method"])
df["shipping_city"] = df["shipping_city"].fillna("Tidak Diketahui")

print("Sisa missing value:")
print(df.isnull().sum())

Sisa missing value:
transaction_id        0
customer_name         0
product_name          0
category              0
price                 0
quantity              0
payment_method        0
transaction_date      0
shipping_city         0
rating              158
dtype: int64


dropna(subset=[...]) menghapus seluruh baris yang memiliki nilai kosong pada kolom-kolom yang disebutkan dalam list. Baris yang customer_name-nya kosong atau payment_method-nya kosong dihapus karena kedua informasi ini wajib ada untuk mengidentifikasi sebuah transaksi

fillna("Tidak Diketahui") mengisi semua nilai kosong pada kolom shipping_city dengan teks "Tidak Diketahui". Strateginya berbeda dengan kolom sebelumnya karena baris yang kota pengirimannya tidak diketahui masih mengandung informasi berguna lainnya seperti harga, kategori, dan tanggal transaksi — sayang jika dibuang.

## K-4 — Deteksi dan Penanganan **Duplicate**

In [6]:
print("Baris duplicate (semua kolom sama):", df.duplicated().sum())
print("transaction_id duplicate:", df['transaction_id'].duplicated().sum())

df = df.drop_duplicates()
print("Jumlah baris setelah drop_duplicates():", len(df))

Baris duplicate (semua kolom sama): 5
transaction_id duplicate: 5
Jumlah baris setelah drop_duplicates(): 490


df.duplicated() menghasilkan series berisi True untuk setiap baris yang merupakan duplikat dari baris sebelumnya (semua kolomnya identik). .sum() menghitung berapa baris yang terduplikat. Baris kedua secara spesifik mengecek duplikat hanya pada kolom transaction_id — untuk memastikan bahwa baris-baris duplikat ini memang merupakan transaksi yang sama persis, bukan transaksi berbeda yang kebetulan punya ID sama.


drop_duplicates() menghapus semua baris yang merupakan duplikat — hanya menyimpan kemunculan pertama dari setiap baris yang identik. Setelah ini jumlah baris seharusnya berkurang dari 500 kembali ke sekitar 485-490 tergantung missing value yang sudah dibuang sebelumnya.

# **K-5 — Koreksi Tipe Data dan Standardisasi Format**

# a. Standardisasi teks kategorikal

In [7]:
for col in ["category", "payment_method", "shipping_city"]:
    df[col] = df[col].astype("string").str.strip().str.title()

# "Cod" adalah singkatan, kembalikan ke huruf kapital penuh
df["payment_method"] = df["payment_method"].replace({"Cod": "COD"})

print(df["category"].value_counts())
print(df["payment_method"].value_counts())

category
Olahraga        97
Kesehatan       91
Elektronik      89
Buku            82
Fashion         66
Rumah Tangga    65
Name: count, dtype: Int64
payment_method
E-Wallet         133
Transfer Bank    125
Kartu Kredit     118
COD              114
Name: count, dtype: Int64


Loop ini menstandardisasi tiga kolom teks sekaligus. .astype("string") mengubah tipe kolom menjadi string pandas — perbedaannya dengan .astype(str) adalah nilai NaN tetap dikenali sebagai missing, bukan berubah menjadi teks "nan". .str.strip() menghapus spasi di awal dan akhir setiap nilai — misalnya "FASHION " menjadi "FASHION". .str.title() mengubah setiap kata menjadi Title Case — huruf pertama kapital, sisanya kecil — sehingga "transfer bank", "TRANSFER BANK", dan "Transfer Bank" semuanya menjadi "Transfer Bank"

Setelah Title Case diterapkan, "COD" berubah menjadi "Cod" karena .str.title() hanya mengkapitalkan huruf pertama. Baris ini mengembalikannya ke "COD" yang benar karena COD adalah singkatan, bukan kata biasa.

# b. Koreksi kolom price

In [8]:
def bersihkan_harga(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip().replace("Rp", "").replace(".", "").replace(",", ".")
    try:
        return float(x)
    except ValueError:
        return np.nan

df["price"] = df["price"].apply(bersihkan_harga)
print(df["price"].dtype)
print(df["price"].head(10))

float64
0       50000.0
1      500000.0
2       25000.0
3      250000.0
5      250000.0
6      120000.0
7     1200000.0
8      150000.0
9     1200000.0
10      50000.0
Name: price, dtype: float64


Fungsi bersihkan_harga dirancang untuk menangani semua variasi format harga yang sengaja dibuat di K-2. Pertama dicek apakah nilainya sudah kosong dengan pd.isna(x) — jika ya, langsung kembalikan np.nan tanpa diproses lebih lanjut. Jika tidak kosong, nilai diubah ke string, spasi dihapus, lalu karakter "Rp", titik ribuan, dan koma dibuang satu per satu. Hasilnya dicoba dikonversi ke float — jika berhasil, nilai numeriknya dikembalikan; jika gagal (karena ada karakter aneh yang tidak terduga), dikembalikan np.nan. .apply(bersihkan_harga) menerapkan fungsi ini ke setiap baris kolom price.

# c. Standardisasi format tanggal ke YYYY-MM-DD

In [9]:
def parse_tanggal(x):
    for fmt in ("%Y-%m-%d", "%d/%m/%Y", "%d-%m-%Y"):
        try:
            return pd.to_datetime(x, format=fmt)
        except ValueError:
            continue
    return pd.NaT

df["transaction_date"] = df["transaction_date"].apply(parse_tanggal).dt.strftime("%Y-%m-%d")
print(df["transaction_date"].head(10))

0     2026-07-14
1     2026-07-10
2     2026-08-26
3     2026-07-28
5     2026-09-16
6     2026-08-08
7     2026-08-18
8     2026-08-19
9     2026-09-12
10    2026-06-19
Name: transaction_date, dtype: object


Fungsi parse_tanggal mencoba tiga format tanggal secara berurutan untuk setiap nilai. Ia mencoba format ISO dulu (%Y-%m-%d), jika gagal mencoba format slash (%d/%m/%Y), jika masih gagal mencoba format dash (%d-%m-%Y). Begitu salah satu format berhasil, nilai tanggal langsung dikembalikan. Jika semua format gagal, dikembalikan pd.NaT (Not a Time — representasi missing untuk data tanggal). Setelah semua tanggal berhasil di-parse, .dt.strftime("%Y-%m-%d") mengubah seluruhnya ke format ISO yang seragam.

# d. Finalisasi tipe data

In [10]:
df["quantity"] = df["quantity"].astype(int)
df["price"] = df["price"].astype(float)

print(df.dtypes)

transaction_id              object
customer_name               object
product_name                object
category            string[python]
price                      float64
quantity                     int64
payment_method      string[python]
transaction_date            object
shipping_city       string[python]
rating                     float64
dtype: object


Dua baris ini memastikan kolom quantity bertipe integer dan kolom price bertipe float — sehingga keduanya bisa langsung digunakan untuk perhitungan matematis tanpa perlu konversi tambahan

# **K-6 — Ekspor Dataset Bersih**

In [11]:
df.to_csv("transaksi_bersih.csv", index=False)
print("Dataset bersih tersimpan:", len(df), "baris")

Dataset bersih tersimpan: 490 baris


df.to_csv("transaksi_bersih.csv", index=False) menyimpan DataFrame yang sudah bersih ke file CSV. Nama file harus persis transaksi_bersih.csv karena file ini akan dibaca ulang di Praktikum 3 dan dikonversi ke format Parquet. index=False memastikan kolom nomor urut baris tidak ikut tersimpan ke file. Jumlah baris akhir yang tersimpan seharusnya 490 baris — ini adalah hasil bersih setelah 15 baris duplikat dan sekitar 10 baris dengan data wajib yang kosong dibuang dari 515 baris awal.